# Pre processing and feature extraction

In this notebook, we are going to explore preprocessing techniques and feature extraction methods that we think that are going to be suitable for the needs of this emotion classification task.


### Imports

In [1]:
import sys
from pathlib import Path
import numpy as np
import pandas as pd
import os
import matplotlib.pyplot as plt
import seaborn as sns
import nltk

# Add project root to path
project_root = Path().absolute().parent
sys.path.append(str(project_root))

from emotion_classifier.data import load_raw_data, prepare_dataset_splits
from emotion_classifier.preprocessing import preprocess_text, clean_text

/opt/miniconda3/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
# Load dataset
train_df, test_df = load_raw_data()
print(f"Loaded dataset with {len(train_df)} samples")

# Create train/validation/test splits if not already done
try:
    train_df, val_df, test_df = prepare_dataset_splits(train_df)
    print(f"Created splits: train={len(train_df)}, val={len(val_df)}, test={len(test_df)}")
except Exception as e:
    print(f"Using existing splits: {e}")
    
# Preview the data
train_df.head()

Loaded dataset with 1874 samples
Using existing splits: Cannot save file into a non-existent directory: '/Users/sofia.avelino/Documents/Emotion_Classifier/data/processed'


,text,primary_emotion,secondary_emotions,meta_emotions,sentiment,interaction_style,intensity,context
0,I’m proud of my friend for overcoming her stru...,Admiration,"['Envy', 'Guilt']","['Pride about admiration', 'Shame about envy']",Mixed,Reflective,4,Relationships
1,Finally finished that project! It was exhausti...,Relief,"['Joy', 'Pride']",['Contentment about relief'],Positive,Assertive,5,Career
2,The laughter we shared during our reunion was ...,Nostalgia,"['Joy', 'Bittersweetness']",['Gratitude about joy'],Positive,Supportive,6,Relationships
3,Why does everything feel so uncertain right no...,Uncertainty,"['Anxiety', 'Frustration']",['Vulnerability about uncertainty'],Ambiguous,Conflicted,5,Self-Reflection
4,Graduation feels both exciting and sad. It’s t...,Bittersweetness,"['Joy', 'Sadness']","['Pride about accomplishment', 'Nostalgia abou...",Mixed,Supportive,6,Education


In [3]:
# 1. Explore basic text cleaning approaches
# Let's examine a sample text before and after cleaning
sample_text = train_df.iloc[10]['text']
print(f"Original text: {sample_text}")

# Basic cleaning (lowercase, remove punctuation and numbers)
cleaned_text = clean_text(sample_text, lower=True, remove_punctuation=True, remove_digits=True)
print(f"Basic cleaning: {cleaned_text}")

# Process with different configurations to compare approaches
tokens_with_stops = preprocess_text(
    sample_text, 
    lower=True, 
    remove_punct=True, 
    remove_digits=True,
    remove_stops=False,
    stemming=False,
    lemmatization=False
)
print(f"After tokenization (with stopwords): {' '.join(tokens_with_stops)}")

tokens_no_stops = preprocess_text(
    sample_text, 
    lower=True, 
    remove_punct=True, 
    remove_digits=True,
    remove_stops=True,
    stemming=False,
    lemmatization=False
)
print(f"After stopword removal: {' '.join(tokens_no_stops)}")

tokens_stemmed = preprocess_text(
    sample_text, 
    lower=True, 
    remove_punct=True, 
    remove_digits=True,
    remove_stops=True,
    stemming=True,
    lemmatization=False
)
print(f"Stemming: {' '.join(tokens_stemmed)}")

tokens_lemmatized = preprocess_text(
    sample_text, 
    lower=True, 
    remove_punct=True, 
    remove_digits=True,
    remove_stops=True,
    stemming=False,
    lemmatization=True
)
print(f"Lemmatization: {' '.join(tokens_lemmatized)}")

Original text: Losing the championship by just one point is heartbreaking, but I’m proud of how far we came as a team.
Basic cleaning: losing the championship by just one point is heartbreaking but i’m proud of how far we came as a team
After tokenization (with stopwords): losing the championship by just one point is heartbreaking but i ’ m proud of how far we came as a team
After stopword removal: losing championship one point heartbreaking ’ proud far came team
Stemming: lose championship one point heartbreak ’ proud far came team
Lemmatization: losing championship one point heartbreaking ’ proud far came team


In [4]:
# 2. Aggregating column values 

#First we start by applying our aggregation to the primary emotions
# Group by primary_emotion and sentiment to count occurrences
emotion_sentiment_counts = train_df.groupby(['primary_emotion', 'sentiment']).size().reset_index(name='count')

# Pivot the data to see the count of each primary_emotion for each sentiment
emotion_sentiment_pivot = emotion_sentiment_counts.pivot_table(index='primary_emotion', 
                                                               columns='sentiment', 
                                                               values='count', 
                                                               fill_value=0)

# Normalize the pivot table by rows
emotion_sentiment_normalized = emotion_sentiment_pivot.div(emotion_sentiment_pivot.sum(axis=1), axis=0)

print(emotion_sentiment_normalized.head())

# Mapping of sentiment categories to numerical values
sentiment_mapping = {"Negative": 0, "Ambiguous": 0.25, "Neutral": 0.5, "Mixed": 0.75, "Positive": 1}

# Function to replace primary_emotion with a corresponding sentiment value
def replace_primary_emotion_with_sentiment(df):
    new_primary_emotion = []
    
    for emotion in df['primary_emotion']:
        if emotion in emotion_sentiment_normalized.index:
            probabilities = emotion_sentiment_normalized.loc[emotion].values
            sentiment_categories = emotion_sentiment_normalized.columns
            
            # Compute cumulative probability intervals
            cumulative_probs = np.cumsum(probabilities)
            random_value = np.random.rand()
            
            # Find the corresponding sentiment
            selected_sentiment = sentiment_categories[np.searchsorted(cumulative_probs, random_value)]
            if (emotion == "Accomplishment"): print(f"Accomplishment is now {selected_sentiment}")
            new_primary_emotion.append(sentiment_mapping[selected_sentiment])
        else:
            # If emotion is not in our probability table, default to neutral
            new_primary_emotion.append(0.5)
    
    return new_primary_emotion

# Apply the transformation to the dataset
train_df['primary_emotion'] = replace_primary_emotion_with_sentiment(train_df)

print(train_df.head())


sentiment        Ambiguous     Mixed  Negative  Neutral  Positive
primary_emotion                                                  
Accomplishment    0.000000  0.000000  0.000000      0.0  1.000000
Admiration        0.295455  0.159091  0.000000      0.0  0.545455
Amazement         0.095238  0.523810  0.000000      0.0  0.380952
Ambivalence       0.125000  0.875000  0.000000      0.0  0.000000
Anger             0.428571  0.190476  0.380952      0.0  0.000000
Accomplishment is now Positive
Accomplishment is now Positive
Accomplishment is now Positive
Accomplishment is now Positive
Accomplishment is now Positive
Accomplishment is now Positive
Accomplishment is now Positive
Accomplishment is now Positive
Accomplishment is now Positive
                                                text  primary_emotion  \
0  I’m proud of my friend for overcoming her stru...             0.75   
1  Finally finished that project! It was exhausti...             1.00   
2  The laughter we shared during our reu

In [5]:
# 2. - Continuation
# Next we aggregate the secondary emotions
# Extract unique emotion pairs from 'secondary_emotions' column
import itertools
pairs = set()

for emotions_list in train_df['secondary_emotions']:
    if isinstance(emotions_list, list) and len(emotions_list) == 2:
        # Sort the emotions in each list so that the order doesn't matter, and store them as a tuple
        pairs.add(tuple(sorted(emotions_list)))

# The total number of unique emotion pairs
total_unique_pairs = len(pairs)

# Display the total number of unique pairs
print(f"Total number of unique emotion pairs: {total_unique_pairs}")


# Process secondary_emotion column
# Explode pairs into individual emotions for probability analysis
train_df['secondary_emotions'] = train_df['secondary_emotions'].astype(str)
train_df_exploded = train_df.assign(secondary_emotion=train_df['secondary_emotions'].str.strip("[]").str.split(', '))
train_df_exploded = train_df_exploded.explode('secondary_emotions')

# Compute sentiment probability distribution for individual secondary emotions
secondary_emotion_sentiment_counts = train_df_exploded.groupby(['secondary_emotions', 'sentiment']).size().reset_index(name='count')
secondary_emotion_sentiment_pivot = secondary_emotion_sentiment_counts.pivot_table(index='secondary_emotions', 
                                                                                     columns='sentiment', 
                                                                                     values='count', 
                                                                                     fill_value=0)
secondary_emotion_sentiment_normalized = secondary_emotion_sentiment_pivot.div(secondary_emotion_sentiment_pivot.sum(axis=1), axis=0)

# Function to compute sentiment for secondary emotion pairs
def replace_secondary_emotion_with_sentiment(df):
    new_secondary_emotion = []
    
    for emotions in df['secondary_emotions']:
        emotions = emotions.strip("[]").split(', ')
        
        if len(emotions) == 2:
            probs1 = secondary_emotion_sentiment_normalized.loc[emotions[0]].values if emotions[0] in secondary_emotion_sentiment_normalized.index else np.array([0.2]*5)
            probs2 = secondary_emotion_sentiment_normalized.loc[emotions[1]].values if emotions[1] in secondary_emotion_sentiment_normalized.index else np.array([0.2]*5)
            
            avg_probs = (probs1 + probs2) / 2  # Average probability
            sentiment_categories = secondary_emotion_sentiment_normalized.columns
            
            cumulative_probs = np.cumsum(avg_probs)
            random_value = np.random.rand()
            
            selected_sentiment = sentiment_categories[np.searchsorted(cumulative_probs, random_value)]
            new_secondary_emotion.append(sentiment_mapping[selected_sentiment])
        else:
            new_secondary_emotion.append(0.5)  # Default to neutral if pair isn't recognized
    
    return new_secondary_emotion

# Apply transformation to secondary_emotion
train_df['secondary_emotions'] = replace_secondary_emotion_with_sentiment(train_df)
print(train_df.head())

Total number of unique emotion pairs: 0
                                                text  primary_emotion  \
0  I’m proud of my friend for overcoming her stru...             0.75   
1  Finally finished that project! It was exhausti...             1.00   
2  The laughter we shared during our reunion was ...             0.75   
3  Why does everything feel so uncertain right no...             0.25   
4  Graduation feels both exciting and sad. It’s t...             0.75   

   secondary_emotions                                      meta_emotions  \
0                0.25     ['Pride about admiration', 'Shame about envy']   
1                0.00                       ['Contentment about relief']   
2                0.50                            ['Gratitude about joy']   
3                0.50                ['Vulnerability about uncertainty']   
4                0.25  ['Pride about accomplishment', 'Nostalgia abou...   

   sentiment interaction_style  intensity          context  
0  

In [6]:
# 2. - Continuation
# Next we aggregate the interaction styles
# Group by interaction_style and sentiment to count occurrences
interaction_sentiment_counts = train_df.groupby(['interaction_style', 'sentiment']).size().reset_index(name='count')

# Pivot the data to see the count of each interaction_style for each sentiment
interaction_sentiment_pivot = interaction_sentiment_counts.pivot_table(index='interaction_style', 
                                                               columns='sentiment', 
                                                               values='count', 
                                                               fill_value=0)

# Normalize the pivot table by rows
interaction_sentiment_normalized = interaction_sentiment_pivot.div(interaction_sentiment_pivot.sum(axis=1), axis=0)

print(interaction_sentiment_normalized.head())

# Mapping of sentiment categories to numerical values
sentiment_mapping = {"Negative": 0, "Ambiguous": 0.25, "Neutral": 0.5, "Mixed": 0.75, "Positive": 1}

# Function to replace interaction_style with a corresponding sentiment value
def replace_interaction_style_with_sentiment(df):
    new_interaction_style = []
    
    for interaction in df['interaction_style']:
        if interaction in interaction_sentiment_normalized.index:
            probabilities = interaction_sentiment_normalized.loc[interaction].values
            sentiment_categories = interaction_sentiment_normalized.columns
            
            # Compute cumulative probability intervals
            cumulative_probs = np.cumsum(probabilities)
            random_value = np.random.rand()
            
            # Find the corresponding sentiment
            selected_sentiment = sentiment_categories[np.searchsorted(cumulative_probs, random_value)]
            new_interaction_style.append(sentiment_mapping[selected_sentiment])
        else:
            # If interaction_style is not in our probability table, default to neutral
            new_interaction_style.append(0.5)
    
    return new_interaction_style

# Apply the transformation to the dataset
train_df['interaction_style'] = replace_interaction_style_with_sentiment(train_df)

print(train_df.head())


sentiment          Ambiguous     Mixed  Negative  Neutral  Positive
interaction_style                                                  
Activating          0.121951  0.268293  0.585366      0.0  0.024390
Adventurous         0.000000  1.000000  0.000000      0.0  0.000000
Affectionate        0.021978  0.076923  0.000000      0.0  0.901099
Analytical          0.500000  0.500000  0.000000      0.0  0.000000
Apologetic          0.000000  0.000000  0.000000      0.0  1.000000
                                                text  primary_emotion  \
0  I’m proud of my friend for overcoming her stru...             0.75   
1  Finally finished that project! It was exhausti...             1.00   
2  The laughter we shared during our reunion was ...             0.75   
3  Why does everything feel so uncertain right no...             0.25   
4  Graduation feels both exciting and sad. It’s t...             0.75   

   secondary_emotions                                      meta_emotions  \
0        

In [7]:
# 2. - Continuation
# Next we aggregate the context values
# Group by context and sentiment to count occurrences
context_sentiment_counts = train_df.groupby(['context', 'sentiment']).size().reset_index(name='count')

# Pivot the data to see the count of each context for each sentiment
context_sentiment_pivot = context_sentiment_counts.pivot_table(index='context', 
                                                               columns='sentiment', 
                                                               values='count', 
                                                               fill_value=0)

# Normalize the pivot table by rows
context_sentiment_normalized = context_sentiment_pivot.div(context_sentiment_pivot.sum(axis=1), axis=0)

print(context_sentiment_normalized.head())

# Mapping of sentiment categories to numerical values
sentiment_mapping = {"Negative": 0, "Ambiguous": 0.25, "Neutral": 0.5, "Mixed": 0.75, "Positive": 1}

# Function to replace context with a corresponding sentiment value
def replace_context_with_sentiment(df):
    new_context_style = []
    
    for context in df['context']:
        if context in context_sentiment_normalized.index:
            probabilities = context_sentiment_normalized.loc[context].values
            sentiment_categories = context_sentiment_normalized.columns
            
            # Compute cumulative probability intervals
            cumulative_probs = np.cumsum(probabilities)
            random_value = np.random.rand()
            
            # Find the corresponding sentiment
            selected_sentiment = sentiment_categories[np.searchsorted(cumulative_probs, random_value)]
            new_context_style.append(sentiment_mapping[selected_sentiment])
        else:
            # If interaction_style is not in our probability table, default to neutral
            new_context_style.append(0.5)
    
    return new_context_style

# Apply the transformation to the dataset
train_df['context'] = replace_context_with_sentiment(train_df)

print(train_df.head())


sentiment             Ambiguous     Mixed  Negative  Neutral  Positive
context                                                               
Achievement            0.000000  0.000000       0.0      0.0  1.000000
Achievement (Sports)   0.000000  0.750000       0.0      0.0  0.250000
Activism               0.000000  0.500000       0.0      0.0  0.500000
Adventure              0.000000  0.666667       0.0      0.0  0.333333
Art                    0.235294  0.294118       0.0      0.0  0.470588
                                                text  primary_emotion  \
0  I’m proud of my friend for overcoming her stru...             0.75   
1  Finally finished that project! It was exhausti...             1.00   
2  The laughter we shared during our reunion was ...             0.75   
3  Why does everything feel so uncertain right no...             0.25   
4  Graduation feels both exciting and sad. It’s t...             0.75   

   secondary_emotions                                      meta_

In [8]:
# 3. Now we want to normalize the intensity values to the interval [0,1]

train_df['intensity'] = train_df['intensity'] / 10
print(train_df.head())

                                                text  primary_emotion  \
0  I’m proud of my friend for overcoming her stru...             0.75   
1  Finally finished that project! It was exhausti...             1.00   
2  The laughter we shared during our reunion was ...             0.75   
3  Why does everything feel so uncertain right no...             0.25   
4  Graduation feels both exciting and sad. It’s t...             0.75   

   secondary_emotions                                      meta_emotions  \
0                0.25     ['Pride about admiration', 'Shame about envy']   
1                0.00                       ['Contentment about relief']   
2                0.50                            ['Gratitude about joy']   
3                0.50                ['Vulnerability about uncertainty']   
4                0.25  ['Pride about accomplishment', 'Nostalgia abou...   

   sentiment  interaction_style  intensity  context  
0      Mixed               0.25        0.4     0.7

In [9]:
# 4. Now we want to begin the process of vectorizing the text
